[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [27]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # Initialize layers
        self.ln1 = nn.LayerNorm(d_model)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.head_size = d_model // num_heads
        self.num_heads = num_heads

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.GELU(),
            nn.Linear(d_model*4, d_model)
        )

    def forward(self, x):
        pass  # Pre-norm + causal attention + MLP with residuals
        x_normed_1 = self.ln1(x)     # (batch_size, seq_len, d_model)

        Q = self.W_q(x_normed_1)     # (batch_size, seq_len, d_model)
        K = self.W_k(x_normed_1)     # (batch_size, seq_len, d_model)
        V = self.W_v(x_normed_1)     # (batch_size, seq_len, d_model)

        attention_scores = []
        for i in range(self.num_heads):
          Q_i = Q[:, :, i*self.head_size : (i+1)*self.head_size]    #(batch_size, seq_len, head_size)
          K_i = K[:, :, i*self.head_size : (i+1)*self.head_size]    #(batch_size, seq_len, head_size)
          V_i = V[:, :, i*self.head_size : (i+1)*self.head_size]    #(batch_size, seq_len, head_size)

          K_i_T = K_i.transpose(1, 2)    #(batch_size, head_size, seq_len)

          QK_i = (Q_i @ K_i_T) / math.sqrt(self.head_size)   #(batch_size, seq_len, seq_len)

          mask = torch.ones_like(QK_i)
          mask = torch.where(torch.tril(mask) == 1, 0, float('-inf'))
          QK_i_masked = QK_i + mask

          softmax_scores_i = torch.softmax(QK_i_masked, dim=-1)    #(batch_size, seq_len, seq_len)
          attention_scores_i = softmax_scores_i @ V_i       #(batch_size, seq_len, head_size)
          attention_scores.append(attention_scores_i)

        attention_scores = torch.cat(attention_scores, dim=-1)   #(batch_size, seq_len, d_model)
        attention_scores = self.W_o(attention_scores)            #(batch_size, seq_len, d_model)

        x = x + attention_scores
        x = x + self.mlp(self.ln2(x))

        return x

In [28]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [29]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (8.0ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (1.2ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.5ms)
  ✅ [4/5] Causal masking — future doesn't affect past (5.3ms)
  ✅ [5/5] Gradient flow to all parameters (6.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (22.2ms total)
  Progress saved. Run status() to see your dashboard.

